In [1]:
import pandas as pd
import pickle

from sklearn.metrics import classification_report, f1_score
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show

In [3]:
STYLE_LABEL_LOOKUP = {
    "style_0": "Festive / Yuletide",
    "style_1": "Epic / Mythic",
    "style_2": "Urban / Street",
    "style_3": "Na na na",
    "style_4": "Nostalgic",
    "style_5": "Romantic / Tender",
    "style_6": "Weathered / Spritual",
    "style_7": "Rastafarian",
    "style_8": "Oh / ooh / la",
    "style_9": "Etheral / Dreamy",
    "style_10": "Somber / Introspective",
    "style_11": "Conversational",
    "style_12": "Uncertain / Restless",
    "style_14": "Extremely Violent"
}
TOPIC_LABEL_LOOKUP = {
    "topic_0": "Existential Reflection",
    "topic_1": "Vocalizations",
    "topic_2": "Old School Gangsta",
    "topic_3": "Americana",
    "topic_4": "Christmas / Reggae",
    "topic_5": "Dark Despair",
    "topic_6": "Desire / Heartbreak",
    "topic_7": "Romantic Love",
    "topic_8": "Nature Imaginary",
    "topic_9": "Hip Hop Slang",
    "topic_10": "Women and Body",
    "topic_11": "Mythic Doom",
    "topic_12": "Wild Fast Living"
}

SELECTED_FEATURES = [
    "style_11",
    "style_6",
    "style_1",
    "style_0",
    "topic_4",
    "style_7",
    "style_14",
    "style_12",
    "style_9",
    "style_2",
    "style_8",
]

In [ ]:
X_train_topics = pd.read_csv("../../data/X_train_topics_dc.csv")
X_train_topics.columns = [f"topic_{col}" for col in X_train_topics.columns]
X_test_topics = pd.read_csv("../../data/X_test_topics_dc.csv")
X_test_topics.columns = [f"topic_{col}" for col in X_test_topics.columns]
X_train_style = pd.read_csv("../../data/X_train_style_dc.csv")
X_train_style.columns = [f"style_{col}" for col in X_train_style.columns]
X_test_style = pd.read_csv("../../data/X_test_style_dc.csv")
X_test_style.columns = [f"style_{col}" for col in X_test_style.columns]
X_train = pd.concat([X_train_topics, X_train_style], axis=1)
X_test = pd.concat([X_test_topics, X_test_style], axis=1)
X_train = X_train[SELECTED_FEATURES]
X_test = X_test[SELECTED_FEATURES]

X_train.rename(columns=STYLE_LABEL_LOOKUP, inplace=True)
X_train.rename(columns=TOPIC_LABEL_LOOKUP, inplace=True)
X_test.rename(columns=STYLE_LABEL_LOOKUP, inplace=True)
X_test.rename(columns=TOPIC_LABEL_LOOKUP, inplace=True)

train_metadata = pd.read_csv("../../data/X_train_metadata_dc.csv")
test_metadata = pd.read_csv("../../data/X_test_metadata_dc.csv")
y_train = train_metadata["dc_detailed"]
y_test = test_metadata["dc_detailed"]

In [17]:
seed = 42
ebm = ExplainableBoostingClassifier(random_state=seed, interactions=3) # not yet supported interactions for multiclass explainability
ebm.fit(X_train, y_train)
with open("experiment_outputs/ebm_with_interactions.pkl", "wb") as f:
    pickle.dump(ebm, f)

N:\Materialien\Promotion\LyricsGenreRecognition\.venv\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:1249: UserWarning: For multiclass we cannot currently visualize pairs and they will be stripped from the global explanations. Set interactions=0 to generate a fully interpretable glassbox model.
  warn(


In [18]:
# with open("experiment_outputs/ebm_test_without_interactions.pkl", "rb") as f:
with open("experiment_outputs/ebm_with_interactions.pkl", "rb") as f:
    ebm = pickle.load(f)
y_pred = ebm.predict(X_test)
print(f"Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred))

Macro F1-Score: 0.2469
                        precision    recall  f1-score   support

                 Blues       0.23      0.03      0.05       656
            Electronic       0.35      0.19      0.25      3975
Folk, World, & Country       0.21      0.01      0.02      1112
           Funk / Soul       0.35      0.19      0.25      2238
               Hip Hop       0.58      0.61      0.60      1507
                  Jazz       0.40      0.11      0.17       955
                 Latin       0.00      0.00      0.00       150
                   Pop       0.25      0.11      0.15      3088
                Reggae       0.51      0.21      0.30       346
                  Rock       0.56      0.90      0.69     11713

              accuracy                           0.51     25740
             macro avg       0.34      0.24      0.25     25740
          weighted avg       0.44      0.51      0.44     25740



N:\Materialien\Promotion\LyricsGenreRecognition\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
N:\Materialien\Promotion\LyricsGenreRecognition\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
N:\Materialien\Promotion\LyricsGenreRecognition\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [16]:
show([ ebm.explain_global(), ebm.explain_local(X_test.iloc[0:5], y_test.iloc[0:5]) ])

N:\Materialien\Promotion\LyricsGenreRecognition\.venv\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:2203: UserWarning: Dropping term Weathered / Spritual & Extremely Violent from explanation since we can't graph multinomial interactions.
  warn(


<!-- http://127.0.0.1:7001/1290219688896/ -->
 Open in new window